In [46]:
# # from abaqus import mdb
# # from abaqusConstants import *
import os
import json
import sys
# # from MESH.meshAlt import *
# # from BCONDITIONS.casing import *
# # from BCONDITIONS.conditions import *
# # from GEOMETRY.geometries import *
# # from JOBS.job import *
from JSONS.ImportTools import *
# # from MATERIALS.materials import *
# # from GEOMETRY.sets import *
# # from GEOMETRY.assembly import *
# from GEOMETRY_PS.geometry_PS import *

path_project = r'C:\Users\leticia\Documents\GitHub\Abaqus_WELL'

if path_project not in sys.path:
    sys.path.append(path_project)

with open(r'C:\Users\leticia\Documents\GitHub\Abaqus_WELL\wellbore_closure_planestrain.json') as f:
    data = json.load(f)
    print(f"Data keys: {data.keys()}")

# Variables read from json (geometry) #####################################

# name_phase = '3dda7930-6dbf-4d05-87f2-d2809a3e9fc6'
# name_tubular = 'LIN_09_875'
if "Phases" not in data["AnalysisData"]:
    print("Chave 'Phases' não encontrada")
    print(data["AnalysisData"].keys())

name_phase = data["AnalysisData"]["Phases"]
print(name_phase)
phase_data = data["Phases"][name_phase]
if phase_data:
    name_tubular = phase_data["Casing"][0]["Tubular"]
else:
    print(f"Phase '{name_phase}' not found in data['Phases']")

############ Rock dimensions ############################
diameter_wellbore = phase_data["HoleDiameter"]
outer_radius_wellbore = diameter_wellbore / 2
outer_radius_wellbore = outer_radius_wellbore * 0.0254  # Convert from inches to meters
thickness_wellbore = outer_radius_wellbore * 0.9  # Variavel da espessura da rocha
inner_radius_wellbore = outer_radius_wellbore - thickness_wellbore

########### Casing / Pipe dimensions ####################
outer_diameter_pipe = data["Tubulars"][name_tubular]['OD']
outer_radius_pipe = (outer_diameter_pipe / 2) * 0.1 # 10% do valor do raio externo para criar um espaço entre a parede do tubo e a borda do modelo
outer_radius_pipe = outer_radius_pipe * 0.0254  # Convert from inches to meters
thickness_pipe = data["Tubulars"][name_tubular]['Thickness']
thickness_pipe = thickness_pipe * 0.1 * 0.0254  # 10% do valor da espessira (inches to meters)
inner_radius_pipe = outer_radius_pipe - thickness_pipe
stand_off = data["AnalysisData"]["StandOff"] / 100   # Convert from inches to meters
min_wall_thickness = data["Tubulars"][name_tubular]["MinThickness"]
# min_wall_thickness = min_wall_thickness / 100  # Convert from inches to meters
# min_wall_thickness = (1 - min_wall_thickness)   # Convert from inches to meters
# thickness_min = thickness_pipe * min_wall_thickness
# ovality = data["Tubulars"][name_tubular]["Ovality"] / 100

########## Annulus dimensions ###########################
outer_radius_annular = inner_radius_wellbore
inner_radius_annular = outer_radius_pipe
thickness_annular = outer_radius_annular - inner_radius_annular

l_depth = data["AnalysisData"]["Depth"]
print(f"The bottom of the wellbore is at: {-l_depth} meters")

lithology = data["Lithology"]
for layer in lithology:
        if l_depth >= layer["Top"] and l_depth < layer["Bottom"]:
            layer_rock = layer["Rock"]
            print(f"Layer at depth {l_depth} meters: {layer_rock}")

Data keys: dict_keys(['AnalysisData', 'ThermalGradient', 'Tubulars', 'Lithology', 'InSituStresses', 'Rocks', 'Cements', 'SteelGrades', 'Phases', 'Events', 'Fluids'])
812c492a-c945-4184-be51-841c5fb86b15
The bottom of the wellbore is at: -2600.0 meters
Layer at depth 2600.0 meters: SANDSTONE


In [50]:
examples = {}

# casing_type = "VM110"
casing_type = data["Tubulars"][name_tubular]["Material"]
print(f"Casing type: {casing_type}") 
# Seleciona o tipo de aço para o casing definido no json (ex: VM-95) e pega as propriedades do material a partir do json
steelgrade_info = data["SteelGrades"][casing_type]
# Seleciona o Gradiente Geotérmico definido no "AnalysisData"
data_geothermal = data["AnalysisData"]["GeothermalGradient"]
print(f"Selected Geothermal Gradient: {data_geothermal}")
# Seleciona o Gradiente Térmico presente e definido antes
thermalGradient = data["ThermalGradient"][data_geothermal]
print(f"Selected Thermal Gradient: {thermalGradient}")
# Retorna uma lista de todos os fluidos com "ThermalGradient" = "data_geothermal"

name_fluido = next(
    (name for name, info in data["Fluids"].items() 
    if info.get("ThermalGradient") == data_geothermal), None)
print(f"Selected Fluid: {name_fluido}")

examples["STEEL"] = {
    "behavior": data["SteelGrades"][casing_type]["Law"],
    'density': data["SteelGrades"][casing_type]["ElasticParameters"]["Density"],
    'elastic': (data["SteelGrades"][casing_type]["ElasticParameters"]["Young"]*1e9,
                data["SteelGrades"][casing_type]["ElasticParameters"]["Poisson"]),
    'conductivity': data["SteelGrades"][casing_type]["ThermalParameters"]["Conductivity"],
    'specific_heat': data["SteelGrades"][casing_type]["ThermalParameters"]["SpecificHeat"],
    'expansion': data["SteelGrades"][casing_type]["ThermalParameters"]["ThermalExpansion"],
    "type": "Casing"
}

examples["FLUID"] = {
    "behavior": "ELASTIC",
    'density': data["Fluids"][name_fluido]["Density"],
    'compressibility': data["Fluids"][name_fluido]["Compressibility"],
    'ThermalExpansion': data["Fluids"][name_fluido]["ThermalExpansion"],
    "type": "Fluid"
}

examples[layer_rock] = {
"behavior": data["Rocks"][layer_rock]["Law"],
'density': data["Rocks"][layer_rock]["ElasticParameters"]["Density"],
'elastic': (data["Rocks"][layer_rock]["ElasticParameters"]["Young"]*1e9,
            data["Rocks"][layer_rock]["ElasticParameters"]["Poisson"]),
'conductivity': data["Rocks"][layer_rock]["ThermalParameters"]["Conductivity"],
'specific_heat': data["Rocks"][layer_rock]["ThermalParameters"]["SpecificHeat"],
'expansion': data["Rocks"][layer_rock]["ThermalParameters"]["ThermalExpansion"],
"type": "Rock"
}

if "MohrCoulombParameters" in data["Rocks"][layer_rock]:
    mc = data["Rocks"][layer_rock]["MohrCoulombParameters"]
    examples[layer_rock].update({
    'friction_angle': mc["FrictionAngle"],
    'dilatancy_angle': mc["DilatancyAngle"],
    'cohesion': mc["Cohesion"],
    "lab_data": ((20001698.76, 0.0), )
    })

if "DoublePowerParameters" in data["Rocks"][layer_rock]:
        examples[layer_rock]["DoublePowerParameters"] = data["Rocks"][layer_rock]["DoublePowerParameters"]

material_examples = {
    "PIPE": {
        "partName": "PIPE",
        "sectionName": 'STEEL_Section',
        "isSolid": True
    },
    "FLUID": {
        "partName": "FLUID",
        "sectionName": 'FLUID_Section',
        "isSolid": True
    }
}

Casing type: K55
Selected Geothermal Gradient: temp. drilling 14in - 60 degC
Selected Thermal Gradient: [{'Depth': 2000.0, 'Temperature': 15.0}, {'Depth': 4550.0, 'Temperature': 60.0}]
Selected Fluid: drilling_14in (60 deg)


In [ ]:
import os
import sys

examples = {}

def ElasticMaterial(modelName, name, data, sectionLength=1.):
    m = mdb.models[modelName]
    mat = m.Material(name=name)
    sect_name = name + '_Section'
    sect = m.HomogeneousSolidSection(name=sect_name,
                                     material=name,
                                     thickness=sectionLength)
    if data.get('density') is not None:
        mat.Density(table=((data.get('density'),),))

    if data.get('elastic') is not None:
        mat.Elastic(table=(data.get('elastic'),))

    if data.get('conductivity') is not None:
        mat.Conductivity(table=((data.get('conductivity'),),))

    if data.get('specific_heat') is not None:
        # raw_value = data.get('specific_heat')
        # corrected_spec_heat = raw_value * 4184.0 
        # mat.SpecificHeat(table=((corrected_spec_heat,),))
        mat.SpecificHeat(table=((data.get('specific_heat'),),))

    if data.get('expansion') is not None:
        mat.Expansion(table=((data.get('expansion'),),))
    subroutine = None
    return mat, sect, subroutine


def vonMisesMaterial(modelName, name, data, sectionLength=1.):
    mat, sect, subroutine = ElasticMaterial(
        modelName, name, data, sectionLength)
    if data.get("stress_strain_curve") is not None:
        mat.Plastic(table=data["stress_strain_curve"])
    return mat, sect, subroutine


def MohrCoulombMaterial(modelName, name, data, sectionLength=1.):
    mat, sect, subroutine = ElasticMaterial(
        modelName, name, data, sectionLength)
    phi = data.get("friction_angle")
    dilat = data.get("dilatancy_angle")
    c = data.get("cohesion")
    labData = data.get("lab_data")
    if None in (phi, c, labData):
        raise ValueError(
            "friction_angle, dilatancy_angle, cohesion, and lab_data must be provided for Mohr-Coulomb material.")
    
    if dilat is None:
        dilat = 0.0
    mat.MohrCoulombPlasticity(table=((phi, dilat), ))
    mat.mohrCoulombPlasticity.MohrCoulombHardening(table=labData)
    mat.mohrCoulombPlasticity.TensionCutOff(temperatureDependency=OFF, dependencies=0,
                                            table=((c, 0.0), ))

    return mat, sect, subroutine

    # mat.Creep(law=USER, table=())


def DoublePowerCreepMaterial(modelName, name, data, sectionLength=1.):
    mat, sect, subroutine = ElasticMaterial(
        modelName, name, data, sectionLength)
    
    print(f"O nome que chegou na função foi: {name}")
    try:
        # dp_data = data["Rocks"][name]["DoublePowerParameters"]        
        dp_data = data.get("DoublePowerParameters", {})
        # dp_data = data.get("creep_parameters", {})
        A1 = dp_data["a1"] 
        A2 = dp_data["a2"] 
        # A1 = dp_data["a1"] / 86400.0  # Convert from per day to per second
        # A2 = dp_data["a2"] / 86400.0  # Convert from per day to per second
        B1 = dp_data["b1"]
        B2 = dp_data["b2"]
        C1 = dp_data["c1"]
        C2 = dp_data["c2"]
        ref_stress = dp_data["s0"]*1e6  # Converting from MPa to Pa
        mat.Creep(law=DOUBLE_POWER,
                  table=((A1, B1, C1, A2, B2, C2, ref_stress),))
    except:
        raise ValueError(
            "double_power_creep_data with A1, A2, B1, B2, C1, C2, and reference_stress must be provided for Double Power Creep material.")
    return mat, sect, subroutine


def DoubleMechanismCreepMaterial(modelName, name, data, sectionLength=1.):
    mat, sect, subroutine = ElasticMaterial(
        modelName, name, data, sectionLength)
    mat.Creep(law=USER, table=())
    subroutine = {"CREEP": " my fortran subroutine "}
    return mat, sect, subroutine

def CreateMaterial(modelName, name, data, sectionLength=1.):
    behavior = data.get("behavior")
    mapping = {
        "ELASTIC": ElasticMaterial,
        "VON_MISES_PLASTIC": vonMisesMaterial,
        "MOHR_COULOMB": MohrCoulombMaterial,
        "DOUBLE_POWER_CREEP": DoublePowerCreepMaterial,
        "DOUBLE_MECHANISM_CREEP": DoubleMechanismCreepMaterial,
    }
    create_func = mapping.get(behavior)
    if create_func is not None:
        return create_func(modelName, name, data, sectionLength)
    else:
        raise ValueError("Behavior '%s' not recognized." % behavior)


def Assign_Section(modelName, partName, sectionName, setName=None, isSolid=True):
    model = mdb.models[modelName]
    # Only work with PIPE and FLUID from material_examples
    allowed_parts = ("PIPE", "FLUID")
    if partName not in allowed_parts:
        print("Skipping section assignment for '%s' (use AssignRockByDepth for rock materials)" % partName)
        return
    if partName not in model.parts:
        raise ValueError("Part '%s' not found in model '%s'." %
                         (partName, modelName))

    part = model.parts[partName]

    default_sets = {
        "FLUID": "FASEI_FLUIDO",
        "PIPE": "FASEI_REV"
    }

    if setName is None:
        setName = default_sets.get(partName, partName)

    if setName in part.sets:
        region = part.sets[setName]
    else:
        if part.space in (TWO_D_PLANAR, AXISYMMETRIC):
            region = part.Set(name=setName, faces=part.faces[:])
        elif part.space == THREE_D:
            region = part.Set(name=setName, cells=part.cells[:])
        # if isSolid is True:
        #     region = part.Set(name=setName, cells=part.cells[:])
        # elif isSolid is False:
        else:
            raise ValueError("No valid entities to assign section in %s" % partName)

    part.SectionAssignment(region=region,
                           sectionName=sectionName,
                           offset=0.0,
                           offsetType=MIDDLE_SURFACE,
                           offsetField='',
                           thicknessAssignment=FROM_SECTION)


def AssignRockByDepth(modelName, partName, rock_layers):
    model = mdb.models[modelName]
    part = model.parts[partName]


    for i, layer in enumerate(rock_layers, start=1):
        sec_name = layer["sectionName"]

        auto_set_name = "L%d-I" % i 
        set_name = layer.get("set_index", auto_set_name)

        if set_name in part.sets:
            region = part.sets[set_name]

            part.SectionAssignment(
                region=region,
                sectionName=sec_name,
                offset=0.0,
                offsetType=MIDDLE_SURFACE,
                offsetField='',
                thicknessAssignment=FROM_SECTION
            )

            print("Assigned section '%s' to existing set '%s'." % (sec_name, set_name))

        else:

            print("Warning: Set '%s' not found in part '%s'. Skipping assignment." % (set_name, partName))




In [ ]:
def ElasticMaterial(modelName, name, data, sectionLength=1.):
    m = mdb.models[modelName]
    mat = m.Material(name=name)
    sect_name = name + '_Section'
    sect = m.HomogeneousSolidSection(name=sect_name,
                                     material=name,
                                     thickness=sectionLength)
    if data.get('density') is not None:
        mat.Density(table=((data.get('density'),),))

    if data.get('elastic') is not None:
        mat.Elastic(table=(data.get('elastic'),))

    if data.get('conductivity') is not None:
        mat.Conductivity(table=((data.get('conductivity'),),))

    if data.get('specific_heat') is not None:
        # raw_value = data.get('specific_heat')
        # corrected_spec_heat = raw_value * 4184.0 
        # mat.SpecificHeat(table=((corrected_spec_heat,),))
        mat.SpecificHeat(table=((data.get('specific_heat'),),))

    if data.get('expansion') is not None:
        mat.Expansion(table=((data.get('expansion'),),))
    subroutine = None
    return mat, sect, subroutine


def vonMisesMaterial(modelName, name, data, sectionLength=1.):
    mat, sect, subroutine = ElasticMaterial(
        modelName, name, data, sectionLength)
    if data.get("stress_strain_curve") is not None:
        mat.Plastic(table=data["stress_strain_curve"])
    return mat, sect, subroutine


def MohrCoulombMaterial(modelName, name, data, sectionLength=1.):
    mat, sect, subroutine = ElasticMaterial(
        modelName, name, data, sectionLength)
    phi = data.get("friction_angle")
    dilat = data.get("dilatancy_angle")
    c = data.get("cohesion")
    labData = data.get("lab_data")
    if None in (phi, c, labData):
        raise ValueError(
            "friction_angle, dilatancy_angle, cohesion, and lab_data must be provided for Mohr-Coulomb material.")
    
    if dilat is None:
        dilat = 0.0
    mat.MohrCoulombPlasticity(table=((phi, dilat), ))
    mat.mohrCoulombPlasticity.MohrCoulombHardening(table=labData)
    mat.mohrCoulombPlasticity.TensionCutOff(temperatureDependency=OFF, dependencies=0,
                                            table=((c, 0.0), ))

    return mat, sect, subroutine

    # mat.Creep(law=USER, table=())


def DoublePowerCreepMaterial(modelName, name, data, sectionLength=1.):
    mat, sect, subroutine = ElasticMaterial(
        modelName, name, data, sectionLength)
    
    print(f"O nome que chegou na função foi: {name}")
    try:
        # dp_data = data["Rocks"][name]["DoublePowerParameters"]        
        dp_data = data.get("DoublePowerParameters", {})
        # dp_data = data.get("creep_parameters", {})
        A1 = dp_data["a1"] 
        A2 = dp_data["a2"] 
        # A1 = dp_data["a1"] / 86400.0  # Convert from per day to per second
        # A2 = dp_data["a2"] / 86400.0  # Convert from per day to per second
        B1 = dp_data["b1"]
        B2 = dp_data["b2"]
        C1 = dp_data["c1"]
        C2 = dp_data["c2"]
        ref_stress = dp_data["s0"]*1e6  # Converting from MPa to Pa
        mat.Creep(law=DOUBLE_POWER,
                  table=((A1, B1, C1, A2, B2, C2, ref_stress),))
    except:
        raise ValueError(
            "double_power_creep_data with A1, A2, B1, B2, C1, C2, and reference_stress must be provided for Double Power Creep material.")
    return mat, sect, subroutine


def DoubleMechanismCreepMaterial(modelName, name, data, sectionLength=1.):
    mat, sect, subroutine = ElasticMaterial(
        modelName, name, data, sectionLength)
    mat.Creep(law=USER, table=())
    subroutine = {"CREEP": " my fortran subroutine "}
    return mat, sect, subroutine

def CreateMaterial(modelName, name, data, sectionLength=1.):
    behavior = data.get("behavior")
    mapping = {
        "ELASTIC": ElasticMaterial,
        "VON_MISES_PLASTIC": vonMisesMaterial,
        "MOHR_COULOMB": MohrCoulombMaterial,
        "DOUBLE_POWER_CREEP": DoublePowerCreepMaterial,
        "DOUBLE_MECHANISM_CREEP": DoubleMechanismCreepMaterial,
    }
    create_func = mapping.get(behavior)
    if create_func is not None:
        return create_func(modelName, name, data, sectionLength)
    else:
        raise ValueError("Behavior '%s' not recognized." % behavior)


def Assign_Section(modelName, partName, sectionName, setName=None, isSolid=True):
    model = mdb.models[modelName]
    # Only work with PIPE and FLUID from material_examples
    allowed_parts = ("PIPE", "FLUID")
    if partName not in allowed_parts:
        print("Skipping section assignment for '%s' (use AssignRockByDepth for rock materials)" % partName)
        return
    if partName not in model.parts:
        raise ValueError("Part '%s' not found in model '%s'." %
                         (partName, modelName))

    part = model.parts[partName]

    default_sets = {
        "FLUID": "FASEI_FLUIDO",
        "PIPE": "FASEI_REV"
    }

    if setName is None:
        setName = default_sets.get(partName, partName)

    if setName in part.sets:
        region = part.sets[setName]
    else:
        if part.space in (TWO_D_PLANAR, AXISYMMETRIC):
            region = part.Set(name=setName, faces=part.faces[:])
        elif part.space == THREE_D:
            region = part.Set(name=setName, cells=part.cells[:])
        # if isSolid is True:
        #     region = part.Set(name=setName, cells=part.cells[:])
        # elif isSolid is False:
        else:
            raise ValueError("No valid entities to assign section in %s" % partName)

    part.SectionAssignment(region=region,
                           sectionName=sectionName,
                           offset=0.0,
                           offsetType=MIDDLE_SURFACE,
                           offsetField='',
                           thicknessAssignment=FROM_SECTION)


def AssignRockByDepth(modelName, partName, rock_layers):
    model = mdb.models[modelName]
    part = model.parts[partName]


    for i, layer in enumerate(rock_layers, start=1):
        sec_name = layer["sectionName"]

        auto_set_name = "L%d-I" % i 
        set_name = layer.get("set_index", auto_set_name)

        if set_name in part.sets:
            region = part.sets[set_name]

            part.SectionAssignment(
                region=region,
                sectionName=sec_name,
                offset=0.0,
                offsetType=MIDDLE_SURFACE,
                offsetField='',
                thicknessAssignment=FROM_SECTION
            )

            print("Assigned section '%s' to existing set '%s'." % (sec_name, set_name))

        else:

            print("Warning: Set '%s' not found in part '%s'. Skipping assignment." % (set_name, partName))



def AddplasticityToSteel(name_model, material_name='STEEL'):
    m = mdb.models[name_model]

    mat = m.materials[material_name]

    plastic_table = (
        (7.58424e+08, 0.0,  273.15),
        (7.58424e+08, 0.25, 273.15),
        (7.56376e+08, 0.0,  298.15),
        (7.56376e+08, 0.25, 298.15),
        (7.25660e+08, 0.0,  373.15),
        (7.25660e+08, 0.25, 373.15),
        (7.05182e+08, 0.0,  423.15),
        (7.05182e+08, 0.25, 423.15),
        (6.84705e+08, 0.0,  473.15),
        (6.84705e+08, 0.25, 473.15),
        (6.64227e+08, 0.0,  523.15),
        (6.64227e+08, 0.25, 523.15)
    )

    mat.Plastic(table=plastic_table, temperatureDependency=ON)

    print(f">>> Plasticity dependent of temperature added to material '{material_name}'!")

for mat_name, mat_data in examples.items():
        CreateMaterial('MyFirstModel', mat_name, mat_data, sectionLength=1.)

mdb.models['MyFirstModel'].setValues(
        absoluteZero=0.0, stefanBoltzmann=5.670374e-8)

AddplasticityToSteel('MyFirstModel', 'STEEL')
